# Activation functions plots

This notebook produces the two figures used in the README (section 2.1):

1. all the activation functions in a single plot
2. all their derivatives in a single plot

Activations are instantiated through `ActivationFunction` from `utils.py`, i.e. with the same parameters used in the experiments (PyTorch defaults). Derivatives are computed with `torch.autograd`, so they are exactly the ones used during backpropagation.

Each function and its derivative share the same color.

In [ ]:
import os
import torch
import matplotlib.pyplot as plt
from utils import ActivationFunction

OUTPUT_DIR = "../images"
os.makedirs(OUTPUT_DIR, exist_ok=True)

## Setup

Activations are listed in the same order of the README. Colors come from a validated categorical palette (first six slots, fixed order).

In [ ]:
# (name used in config.yaml, label shown in the plots, color)
ACTIVATIONS = [
    ("relu",  "ReLU",         "#2a78d6"),
    ("lrelu", "Leaky ReLU",   "#eb6834"),
    ("elu",   "ELU",          "#1baf7a"),
    ("gelu",  "GELU",         "#eda100"),
    ("selu",  "SELU",         "#e87ba4"),
    ("swish", "Swish (SiLU)", "#008300"),
]

SURFACE, INK, INK_SOFT, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#e3e2de"

# 1000 points on [-5, 5]: x = 0 is not sampled, so no derivative convention at 0 shows up in the plots
x = torch.linspace(-5, 5, 1000)

## Computing functions and derivatives

In [ ]:
factory = ActivationFunction()
curves = {}

for name, label, color in ACTIVATIONS:
    act = factory(name)
    xi = x.clone().requires_grad_(True)
    yi = act(xi)
    yi.sum().backward()          # elementwise function -> xi.grad[k] = f'(x_k)
    curves[name] = (yi.detach().numpy(), xi.grad.numpy())
    print("{:<14} {}".format(label, act))

## Plotting

For $x > 0$ several functions (and derivatives) almost coincide, so lines are drawn with decreasing widths: the ones underneath remain visible as a colored border.

In [ ]:
WIDTHS = [5.0, 4.2, 3.4, 2.6, 2.0, 1.4]

def plot_curves(which, ylabel, title, ylim, filename):
    fig, ax = plt.subplots(figsize=(9, 6), facecolor=SURFACE)
    ax.set_facecolor(SURFACE)

    for (name, label, color), lw in zip(ACTIVATIONS, WIDTHS):
        ax.plot(x.numpy(), curves[name][which], color=color, linewidth=lw,
                label=label, solid_capstyle="round")

    ax.axhline(0, color=INK_SOFT, linewidth=0.8)
    ax.axvline(0, color=INK_SOFT, linewidth=0.8)
    ax.set_xlim(-5, 5)
    ax.set_ylim(*ylim)
    ax.grid(True, color=GRID, linewidth=0.8)
    ax.set_axisbelow(True)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    for side in ("left", "bottom"):
        ax.spines[side].set_color(INK_SOFT)

    ax.set_xlabel("Input x", color=INK_SOFT, fontsize=12)
    ax.set_ylabel(ylabel, color=INK_SOFT, fontsize=12)
    ax.set_title(title, color=INK, fontsize=15, fontweight="bold", loc="left", pad=14)
    ax.tick_params(colors=INK_SOFT, labelsize=10)

    legend = ax.legend(loc="upper left", frameon=True, fontsize=11, title="Activation")
    legend.get_frame().set_facecolor(SURFACE)
    legend.get_frame().set_edgecolor(GRID)
    legend.get_title().set_color(INK_SOFT)
    for text in legend.get_texts():
        text.set_color(INK)

    fig.tight_layout()
    fig.savefig(os.path.join(OUTPUT_DIR, filename), dpi=150, facecolor=SURFACE)
    plt.show()

### Activation functions

In [ ]:
plot_curves(which=0, ylabel="f(x)", title="Activation functions",
            ylim=(-2.5, 5), filename="all_activations.png")

### Derivatives

In [ ]:
plot_curves(which=1, ylabel="f'(x)", title="Derivatives of the activation functions",
            ylim=(-0.15, 1.9), filename="all_derivatives.png")